# Cleaning bayut_properties.csv

In [2]:
import pandas as pd
import numpy as np
import ast

df = pd.read_csv("bayut_properties.csv")
df.shape

(449, 19)

In [3]:
df.head()

,url,reference_number,id,broker_display_name,title,property_type,description,location,price,currency,bedrooms,bathrooms,furnished,amenities,details,agent_name,property_image_urls,completion_status,ownership
0,https://www.bayut.eg/en/property/details-50397...,Bayut - 042015,503976030,Remax Avalon,Apartment for sale 165 m Smouha (Sh Adamon Fre...,Apartment,"شقة للبيع 165 م سموحه ( ش ادمون فريمون ) - 5,0...","Smoha, Alexandria","5,000,000",EGP,4,1.0,Unfurnished,NaN,165 Sq. M.,Aly Sallam,"[""https://images.bayut.eg/thumbnails/48290666-...",Ready,Resale
1,https://www.bayut.eg/en/property/details-50403...,Bayut - 1412-oZqhry,504034847,Property Hills One 1,"A villa overlooking a landscaped area, ready f...",Villa,A fully finished villa with an ultra-superb fi...,"Sodic East, New Heliopolis, Cairo","33,000,000",EGP,4,4.0,Furnished,"Built in Kitchen Appliances, Furnished, Covere...",331 Sq. M.,Santi Tareq,"[""https://images.bayut.eg/thumbnails/49130472-...",Ready,Primary
2,https://www.bayut.eg/en/property/details-50403...,Bayut - 1412-KXhp1U,504034783,Property Hills One 1,The last villa in the compound is ready for vi...,Villa,The last villa in the compound is ready for vi...,"Ivy Residence, Shorouk City, Cairo","15,000,000",EGP,5,5.0,Unfurnished,NaN,376 Sq. M.,merna elkady,"[""https://images.bayut.eg/thumbnails/49130137-...",Ready,Primary
3,https://www.bayut.eg/en/property/details-50403...,Bayut - 1016-lNhXuj,504034767,Falcon Real Estate,Townhouse with view of Central Park in La Vist...,Townhouse,with 25% cash discount prime Townhouse with op...,"Patio Town Compound, 5th Settlement, New Cairo...","22,500,000",EGP,4,4.0,NaN,NaN,223 Sq. M.,abdelrahman mohamed elsaidy,"[""https://images.bayut.eg/thumbnails/49129847-...",Ready,Primary
4,https://www.bayut.eg/en/property/details-50403...,Bayut - 1016-uZZby7,504034491,Falcon Real Estate,Duplex 206m + Garden Open View Opposite Madina...,Apartment,206m² Private Garden Duplex | Prime Madinaty V...,"Sarai, Mostakbal City, Cairo","8,200,000",EGP,4,4.0,Unfurnished,"Gym or Health Club, Swimming Pool, Jacuzzi, Sa...",206 Sq. M.,Dina Mohamed,"[""https://images.bayut.eg/thumbnails/49126818-...",Off-Plan,Primary


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 449 entries, 0 to 448
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   url                  449 non-null    str    
 1   reference_number     449 non-null    str    
 2   id                   449 non-null    int64  
 3   broker_display_name  449 non-null    str    
 4   title                449 non-null    str    
 5   property_type        449 non-null    str    
 6   description          449 non-null    str    
 7   location             449 non-null    str    
 8   price                449 non-null    str    
 9   currency             449 non-null    str    
 10  bedrooms             449 non-null    int64  
 11  bathrooms            448 non-null    float64
 12  furnished            387 non-null    str    
 13  amenities            255 non-null    str    
 14  details              449 non-null    str    
 15  agent_name           449 non-null    str    
 16  p

## Checking what's wrong with it

- `price` is a string with commas in it (`"5,000,000"`) instead of a number
- `details` has the area buried inside text like `"165 Sq. M."`
- `amenities` is one long comma separated string, and it's blank for a lot of rows
- `property_image_urls` looks like a list but it's actually just a string
- `location` mixes area + city together in one string
- `currency` is EGP for literally every row, so it's not adding anything
- `furnished`, `ownership`, `bathrooms` have some missing values
- `description` has leftover junk like `[redacted phone number]` and weird spacing from scraping

In [5]:
df.isna().sum()

url                      0
reference_number         0
id                       0
broker_display_name      0
title                    0
property_type            0
description              0
location                 0
price                    0
currency                 0
bedrooms                 0
bathrooms                1
furnished               62
amenities              194
details                  0
agent_name               0
property_image_urls      0
completion_status        0
ownership                2
dtype: int64

In [6]:
df.shape

(449, 19)

In [7]:
df['currency'].value_counts()

currency
EGP    449
Name: count, dtype: int64

In [8]:
df.duplicated(subset=['id']).sum()

np.int64(0)

## Fixing price

Strip the commas and convert to a proper number.

In [9]:
df['price'] = df['price'].str.replace(',', '', regex=False).astype(float)
df['price'].describe()

count    4.490000e+02
mean     1.700014e+07
std      2.201048e+07
min      1.600000e+06
25%      6.000000e+06
50%      1.025000e+07
75%      2.000000e+07
max      2.600000e+08
Name: price, dtype: float64

## Pulling area out of `details`

Every value looks like `"165 Sq. M."` so I can just grab the number.

In [10]:
df['area_sqm'] = df['details'].str.extract(r'([\d.]+)').astype(float)
df[['details', 'area_sqm']].head()

,details,area_sqm
0,165 Sq. M.,165.0
1,331 Sq. M.,331.0
2,376 Sq. M.,376.0
3,223 Sq. M.,223.0
4,206 Sq. M.,206.0


## Splitting location into area / city

The last comma-separated piece is basically always the city, everything before it is the area/compound.

In [11]:
loc_split = df['location'].str.split(',')
df['city'] = loc_split.str[-1].str.strip()
df['area'] = loc_split.apply(lambda parts: ','.join(p.strip() for p in parts[:-1]))
df[['location', 'area', 'city']].head()

,location,area,city
0,"Smoha, Alexandria",Smoha,Alexandria
1,"Sodic East, New Heliopolis, Cairo","Sodic East,New Heliopolis",Cairo
2,"Ivy Residence, Shorouk City, Cairo","Ivy Residence,Shorouk City",Cairo
3,"Patio Town Compound, 5th Settlement, New Cairo...","Patio Town Compound,5th Settlement,New Cairo",Cairo
4,"Sarai, Mostakbal City, Cairo","Sarai,Mostakbal City",Cairo


In [12]:
df['city'].value_counts().head(10)

city
Cairo         229
Matruh        128
Giza           47
Alexandria     15
Red Sea        15
Suez           12
Sharqia         1
Damietta        1
Sinai           1
Name: count, dtype: int64

## Amenities -> list + count

Instead of one messy string, split it into an actual list and also keep a count (easier to filter/sort on).

In [13]:
df['amenities_list'] = df['amenities'].fillna('').apply(
    lambda x: [a.strip() for a in x.split(',') if a.strip()]
)
df['amenities_count'] = df['amenities_list'].apply(len)
df[['amenities_count']].describe()

,amenities_count
count,449.000000
mean,12.917595
std,15.255411
min,0.000000
25%,0.000000
50%,9.000000
75%,20.000000
max,51.000000


## Fixing property_image_urls

This is stored as a string that *looks* like a Python list, so `ast.literal_eval` turns it into a real list. I'll also just keep an image count since that's usually more useful than the full list.

In [14]:
df['property_image_urls'] = df['property_image_urls'].apply(ast.literal_eval)
df['image_count'] = df['property_image_urls'].apply(len)
df[['image_count']].describe()

,image_count
count,449.000000
mean,12.510022
std,4.602068
min,2.000000
25%,10.000000
50%,11.000000
75%,14.000000
max,43.000000


## Cleaning up the description text

Get rid of the `[redacted phone number]` placeholders and some scraping artifacts, and collapse extra whitespace. Not translating anything, just tidying it.

In [15]:
def clean_description(text):
    text = str(text)
    text = text.replace('[redacted phone number]', '')
    text = text.replace('. me/', '.me/')
    text = ' '.join(text.split())  # collapse extra spaces
    return text

df['description'] = df['description'].apply(clean_description)
df['description'].head(3)

0    شقة للبيع 165 م سموحه ( ش ادمون فريمون ) - 5,0...
1    A fully finished villa with an ultra-superb fi...
2    The last villa in the compound is ready for vi...
Name: description, dtype: str

## Missing values

`furnished` and `ownership` are missing for a chunk of rows, and there's 1 missing `bathrooms`. I don't want to guess furnishing/ownership status, so I'll just label them 'Unknown' instead of leaving NaN. For bathrooms, filling with the median for that property_type feels safer than dropping the row.

In [16]:
df['furnished'] = df['furnished'].fillna('Unknown')
df['ownership'] = df['ownership'].fillna('Unknown')

df['bathrooms'] = df.groupby('property_type')['bathrooms'].transform(
    lambda x: x.fillna(x.median())
)

df[['furnished', 'ownership', 'bathrooms']].isna().sum()

furnished    0
ownership    0
bathrooms    1
dtype: int64

## Dropping columns that aren't useful anymore

`currency` is always EGP so it's dead weight, and `details` has been replaced by `area_sqm`.

In [17]:
df = df.drop(columns=['currency', 'details'])
df.columns.tolist()

['url',
 'reference_number',
 'id',
 'broker_display_name',
 'title',
 'property_type',
 'description',
 'location',
 'price',
 'bedrooms',
 'bathrooms',
 'furnished',
 'amenities',
 'agent_name',
 'property_image_urls',
 'completion_status',
 'ownership',
 'area_sqm',
 'city',
 'area',
 'amenities_list',
 'amenities_count',
 'image_count']

## Final check

In [18]:
df.isna().sum()

url                      0
reference_number         0
id                       0
broker_display_name      0
title                    0
property_type            0
description              0
location                 0
price                    0
bedrooms                 0
bathrooms                1
furnished                0
amenities              194
agent_name               0
property_image_urls      0
completion_status        0
ownership                0
area_sqm                 0
city                     0
area                     0
amenities_list           0
amenities_count          0
image_count              0
dtype: int64

In [19]:
df.head()

,url,reference_number,id,broker_display_name,title,property_type,description,location,price,bedrooms,...,agent_name,property_image_urls,completion_status,ownership,area_sqm,city,area,amenities_list,amenities_count,image_count
0,https://www.bayut.eg/en/property/details-50397...,Bayut - 042015,503976030,Remax Avalon,Apartment for sale 165 m Smouha (Sh Adamon Fre...,Apartment,"شقة للبيع 165 م سموحه ( ش ادمون فريمون ) - 5,0...","Smoha, Alexandria",5000000.0,4,...,Aly Sallam,[https://images.bayut.eg/thumbnails/48290666-8...,Ready,Resale,165.0,Alexandria,Smoha,[],0,10
1,https://www.bayut.eg/en/property/details-50403...,Bayut - 1412-oZqhry,504034847,Property Hills One 1,"A villa overlooking a landscaped area, ready f...",Villa,A fully finished villa with an ultra-superb fi...,"Sodic East, New Heliopolis, Cairo",33000000.0,4,...,Santi Tareq,[https://images.bayut.eg/thumbnails/49130472-8...,Ready,Primary,331.0,Cairo,"Sodic East,New Heliopolis","[Built in Kitchen Appliances, Furnished, Cover...",41,12
2,https://www.bayut.eg/en/property/details-50403...,Bayut - 1412-KXhp1U,504034783,Property Hills One 1,The last villa in the compound is ready for vi...,Villa,The last villa in the compound is ready for vi...,"Ivy Residence, Shorouk City, Cairo",15000000.0,5,...,merna elkady,[https://images.bayut.eg/thumbnails/49130137-8...,Ready,Primary,376.0,Cairo,"Ivy Residence,Shorouk City",[],0,10
3,https://www.bayut.eg/en/property/details-50403...,Bayut - 1016-lNhXuj,504034767,Falcon Real Estate,Townhouse with view of Central Park in La Vist...,Townhouse,with 25% cash discount prime Townhouse with op...,"Patio Town Compound, 5th Settlement, New Cairo...",22500000.0,4,...,abdelrahman mohamed elsaidy,[https://images.bayut.eg/thumbnails/49129847-8...,Ready,Primary,223.0,Cairo,"Patio Town Compound,5th Settlement,New Cairo",[],0,20
4,https://www.bayut.eg/en/property/details-50403...,Bayut - 1016-uZZby7,504034491,Falcon Real Estate,Duplex 206m + Garden Open View Opposite Madina...,Apartment,206m² Private Garden Duplex | Prime Madinaty V...,"Sarai, Mostakbal City, Cairo",8200000.0,4,...,Dina Mohamed,[https://images.bayut.eg/thumbnails/49126818-8...,Off-Plan,Primary,206.0,Cairo,"Sarai,Mostakbal City","[Gym or Health Club, Swimming Pool, Jacuzzi, S...",11,10


## Save cleaned file

In [20]:
df.to_csv('bayut_properties_cleaned.csv', index=False)
print("saved:", df.shape)

saved: (449, 23)
